# Notebook 2 — Per-patch IQA metric extraction

**Goal:** compute no-reference quality metrics over the whole ARCADE dataset,
assessing each image **locally** through a grid of patches.

**Metrics:**

| Metric | Type | Rationale from the literature |
|---|---|---|
| Laplacian variance | Sharpness | Standard, robust on radiographs |
| RMS contrast | Local contrast | Sensitive to motion artefacts |
| Shannon entropy | Disorder / artefacts | Validated on DSA (PatchDSA, 2024) |
| Tenengrad | Global gradient | Robust to the thin edges of the arteries |
| Directional Tenengrad (H/V/D1/D2) | Analysis in 4 directions | MSCN paper (2024) — tubular structure |

**Deliverable:** `outputs/patch_metrics_full.csv` + 2D quality maps per image.

## 0. Configuration

In [ ]:
import os
from pathlib import Path
from pipeline_config import use_resnet, APPROACH

# ── This notebook is INDEPENDENT of the approach (A/B/C). ──
# It extracts the per-patch IQA metrics for all images.
# Only USE_RESNET depends on the approach (needed for C only).
# No need to rerun this notebook when the approach changes (except A/B → C).

# Paths — dataset: $ARCADE_ROOT if set, otherwise <repo>/data/arcade
PROJECT_ROOT   = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
ARCADE_ROOT    = Path(os.environ.get("ARCADE_ROOT", PROJECT_ROOT / "data" / "arcade"))
SEG_TRAIN_PATH = ARCADE_ROOT / "dataset_phase_1/segmentation_dataset/seg_train"
SEG_VAL_PATH   = ARCADE_ROOT / "dataset_phase_1/segmentation_dataset/seg_val"

OUTPUT_DIR        = Path("outputs_cvae/02_IQA_Metrics_Extraction")
FIGURES_DIR       = OUTPUT_DIR / "figures"
METRICS_DIR       = OUTPUT_DIR / "metrics"
RESNET_FEAT_DIR   = OUTPUT_DIR / "resnet_features"
QUALITY_MAPS_DIR  = OUTPUT_DIR / "quality_maps"
for _d in [OUTPUT_DIR, FIGURES_DIR, METRICS_DIR, RESNET_FEAT_DIR, QUALITY_MAPS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

GRID_SIZE    = 4      # grille G×G (4→16 patchs, 8→64 patchs)
MSCN_WINDOW  = 7      # window size of the MSCN normalisation
N_WORKERS    = 4      # threads for parallel processing
USE_RESNET   = use_resnet()  # True for approach C only

print(f"Grid: {GRID_SIZE}×{GRID_SIZE} = {GRID_SIZE**2} patches per image")
print(f"ResNet feature extraction : {USE_RESNET}")
print(f"\nNote: this notebook extracts the raw IQA metrics, independently of the approach.")

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from scipy.ndimage import uniform_filter, convolve
from skimage.measure import shannon_entropy
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import random, warnings

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

print("Imports OK")

## 2. Loading the paths (whole dataset)

In [ ]:
def collect_images(root: Path, exts=(".png", ".jpg", ".jpeg")) -> list[Path]:
    image_dir = root / "images" if (root / "images").exists() else root
    return sorted(p for p in image_dir.rglob("*") if p.suffix.lower() in exts)

all_paths = collect_images(SEG_TRAIN_PATH) + collect_images(SEG_VAL_PATH)
print(f"Dataset total : {len(all_paths)} images")

# ⚠ Fixed bug: train and val are numbered independently (1.png..N.png in
# each folder), so path.stem alone collides between the two splits (e.g.
# seg_train/images/1.png and seg_val/images/1.png share the "1"). The split is used
# as a prefix so that image_id is unique over the whole dataset.
def make_image_id(path: Path) -> str:
    split = "train" if "seg_train" in str(path) else ("val" if "seg_val" in str(path) else "unk")
    return f"{split}_{path.stem}"


## 3. Utility functions (taken from notebook 1)

In [ ]:
def load_gray(path: Path) -> np.ndarray:
    """Load a greyscale image, float32 in [0, 1]."""
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0


def mscn_normalize(img: np.ndarray, window: int = MSCN_WINDOW, C: float = 1.0) -> np.ndarray:
    """MSCN normalisation: removes the local illumination biases."""
    img64 = img.astype(np.float64)
    mu    = uniform_filter(img64, size=window)
    sigma = np.sqrt(np.maximum(uniform_filter(img64 ** 2, size=window) - mu ** 2, 0))
    return ((img64 - mu) / (sigma + C)).astype(np.float32)

## 4. Patch extraction

An image of size H×W is split into `GRID_SIZE × GRID_SIZE` blocks of size `(H//G, W//G)`.
Residual pixels at the image border are ignored (clean crop).

In [ ]:
def extract_patches(img: np.ndarray, G: int = GRID_SIZE) -> np.ndarray:
    """
    Split img into a G×G grid of patches.
    Returns an array of shape (G, G, ph, pw).
    """
    H, W  = img.shape
    ph, pw = H // G, W // G
    patches = np.zeros((G, G, ph, pw), dtype=img.dtype)
    for i in range(G):
        for j in range(G):
            patches[i, j] = img[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
    return patches


# Quick check
test_img = load_gray(all_paths[0])
patches  = extract_patches(test_img)
print(f"Image: {test_img.shape}  →  Patches: {patches.shape}  (each patch: {patches.shape[2]}×{patches.shape[3]})")

## 5. Per-patch metrics

### 5a. Classical scalar metrics

In [ ]:
from scipy.ndimage import laplace

def patch_laplacian_var(p: np.ndarray) -> float:
    return float(laplace(p).var())

def patch_rms_contrast(p: np.ndarray) -> float:
    return float(p.std())

def patch_entropy(p: np.ndarray) -> float:
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-8)
    return float(shannon_entropy(p_norm))

def patch_tenengrad(p: np.ndarray) -> float:
    """L2 Sobel gradient — sharpness of the edges."""
    from scipy.ndimage import sobel
    Gx = sobel(p, axis=1)
    Gy = sobel(p, axis=0)
    return float(np.sqrt(Gx**2 + Gy**2).mean())

## 9. Feature extraction with ResNet-18

If the signal-processing metrics are not enough to separate the images,
a pre-trained ResNet-18 can serve as a **frozen texture extractor** (no fine-tuning needed).

The vector of the `AdaptiveAvgPool2d` layer (dim 512) encodes the global texture of each image.
It can serve as a richer inductive bias for the cVAE (notebook 4).

**Enabled automatically when `APPROACH = "C"` in `pipeline_config.py`.**

In [ ]:
SOBEL_H  = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
SOBEL_V  = SOBEL_H.T
SOBEL_D1 = np.array([[ 0,  1,  2], [-1,  0,  1], [-2, -1,  0]], dtype=np.float32)
SOBEL_D2 = np.array([[ 2,  1,  0], [ 1,  0, -1], [ 0, -1, -2]], dtype=np.float32)

_KERNELS = {"H": SOBEL_H, "V": SOBEL_V, "D1": SOBEL_D1, "D2": SOBEL_D2}

def patch_directional_tenengrad(p: np.ndarray) -> dict[str, float]:
    """
    Compute the mean gradient response in 4 directions.
    Returns a dict {H, V, D1, D2}.
    """
    return {name: float(np.abs(convolve(p, k)).mean())
            for name, k in _KERNELS.items()}


# Test on one patch
p0 = patches[0, 0]
print(f"Tenengrad global : {patch_tenengrad(p0):.5f}")
print(f"Tenengrad directionnel : {patch_directional_tenengrad(p0)}")

## 6. Quality map of an image

All the metrics are combined into a **quality map** of shape `(G, G, n_metrics)`.

In [ ]:
METRIC_NAMES = ["laplacian_var", "rms_contrast", "entropy", "tenengrad", "ten_H", "ten_V", "ten_D1", "ten_D2"]

def compute_quality_map(img: np.ndarray, G: int = GRID_SIZE) -> np.ndarray:
    """
    Return a (G, G, 8) array with the metrics of each patch.
    The image must already be normalised (MSCN or raw).
    """
    patches = extract_patches(img, G)
    qmap    = np.zeros((G, G, len(METRIC_NAMES)), dtype=np.float32)

    for i in range(G):
        for j in range(G):
            p   = patches[i, j]
            dt  = patch_directional_tenengrad(p)
            qmap[i, j] = [
                patch_laplacian_var(p),
                patch_rms_contrast(p),
                patch_entropy(p),
                patch_tenengrad(p),
                dt["H"], dt["V"], dt["D1"], dt["D2"],
            ]
    return qmap


# Demo on the first image, MSCN-normalised
demo_img  = mscn_normalize(load_gray(all_paths[0]))
demo_qmap = compute_quality_map(demo_img)
print(f"Quality map: {demo_qmap.shape}  (G={GRID_SIZE}, {len(METRIC_NAMES)} metrics)")

## 7. Visualisation: image + quality heatmaps

In [ ]:
def plot_quality_maps(img_raw: np.ndarray, qmap: np.ndarray, title: str = "", save_path=None):
    n_metrics = len(METRIC_NAMES)
    fig, axes = plt.subplots(2, (n_metrics // 2) + 1, figsize=(18, 7))
    axes = axes.flatten()

    # Image originale
    axes[0].imshow(img_raw, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Image brute", fontsize=9)
    axes[0].axis("off")

    # One heatmap per metric
    for k, name in enumerate(METRIC_NAMES):
        data = qmap[:, :, k]
        ax   = axes[k + 1]
        im   = ax.imshow(data, cmap="RdYlGn", aspect="auto",
                         vmin=np.percentile(data, 5), vmax=np.percentile(data, 95))
        ax.set_title(name, fontsize=8)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for j in range(n_metrics + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()


plot_quality_maps(
    load_gray(all_paths[0]),
    demo_qmap,
    title=f"Quality maps — {all_paths[0].stem} ({GRID_SIZE}×{GRID_SIZE} grid)",
    save_path=FIGURES_DIR / "02_qmap_demo.png",
)

## 8. Directional analysis: the signature of cardiac motion blur

A heartbeat artefact mainly degrades the **vertical** direction (the heart contracts from top to bottom).
The difference in response between the 4 directions is shown to tell artefacted images from sharp ones.

In [ ]:
# Load a few varied images
demo_paths = random.sample(all_paths, min(6, len(all_paths)))

fig, axes = plt.subplots(len(demo_paths), 4, figsize=(14, 3 * len(demo_paths)))
dir_labels = ["ten_H", "ten_V", "ten_D1", "ten_D2"]
dir_idx    = [4, 5, 6, 7]  # indices in METRIC_NAMES

for row, path in enumerate(demo_paths):
    img  = mscn_normalize(load_gray(path))
    qmap = compute_quality_map(img)
    vmin = qmap[:, :, dir_idx].min()
    vmax = qmap[:, :, dir_idx].max()

    for col, (label, idx) in enumerate(zip(dir_labels, dir_idx)):
        ax = axes[row, col]
        im = ax.imshow(qmap[:, :, idx], cmap="hot", vmin=vmin, vmax=vmax)
        if row == 0:
            ax.set_title(label, fontsize=9)
        if col == 0:
            ax.set_ylabel(path.stem[:18], fontsize=7)
        ax.axis("off")

fig.suptitle("Directional Tenengrad — comparison between images", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_directional_tenengrad.png", dpi=100, bbox_inches="tight")
plt.show()

## 9. Feature extraction with ResNet-18

If the signal-processing metrics are not enough to separate the images,
a pre-trained ResNet-18 can serve as a **frozen texture extractor** (no fine-tuning needed).

The vector of the `AdaptiveAvgPool2d` layer (dim 512) encodes the global texture of each image.
It can serve as a richer inductive bias for the cVAE (notebook 4).

**Enabled automatically when `APPROACH = "C"` in `pipeline_config.py`.**

In [ ]:
if USE_RESNET:
    import torch
    import torchvision.models as models
    import torchvision.transforms.functional as TF

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device : {device}")

    resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # Remove the classification head, keep up to AdaptiveAvgPool2d
    feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()

    print(f"Feature extractor ready — output dim: 512")
else:
    print(f"ResNet disabled (approach {APPROACH} does not need it)")

In [ ]:
if USE_RESNET:
    def extract_resnet_features(path: Path) -> np.ndarray:
        """
        Returns a 512-dimensional ResNet-18 feature vector.
        The greyscale image is duplicated over 3 channels to match the ResNet input.
        """
        img = load_gray(path)  # (H, W) float32
        # → (1, 1, H, W) puis expand vers (1, 3, H, W)
        t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).expand(-1, 3, -1, -1)
        # Resize to 224×224 (expected by ResNet)
        t = TF.resize(t, [224, 224])
        # Normalisation ImageNet
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        t    = (t - mean) / std
        with torch.no_grad():
            feat = feature_extractor(t.to(device))  # (1, 512, 1, 1)
        return feat.squeeze().cpu().numpy()  # (512,)

    # Test
    test_feat = extract_resnet_features(all_paths[0])
    print(f"Vecteur features : {test_feat.shape}  — mean={test_feat.mean():.4f}")

## 10. Processing the whole dataset

The metrics are computed for **all** images and the patches are aggregated into global scores per image.

In [ ]:
# Intensity threshold to ignore "black" patches (radiograph borders)
PATCH_BLACK_THRESHOLD = 0.02   # mean pixel < 5/255 ≈ 0.02

def process_one_image(path: Path) -> dict:
    """
    Process an image and return a dict of global metrics.
    - Patches with a mean intensity < PATCH_BLACK_THRESHOLD are ignored.
    - Exports mean, std, max and min per metric (over the valid patches).
    """
    try:
        raw  = load_gray(path)                     # [0, 1]
        img  = mscn_normalize(raw)
        qmap = compute_quality_map(img)            # (G, G, 8)

        # Mask: non-black patches (computed on the raw image)
        raw_patches = extract_patches(raw, GRID_SIZE)  # (G, G, ph, pw)
        patch_means = raw_patches.mean(axis=(2, 3))    # (G, G)
        valid_mask  = patch_means >= PATCH_BLACK_THRESHOLD  # (G, G) bool

        record = {"image_id": make_image_id(path), "path": str(path)}
        n_valid = int(valid_mask.sum())
        record["n_valid_patches"] = n_valid

        if n_valid > 0:
            # Reshape to filter: (G*G, 8)
            flat = qmap.reshape(-1, len(METRIC_NAMES))
            mask = valid_mask.ravel()
            valid_vals = flat[mask]  # (n_valid, 8)

            for k, name in enumerate(METRIC_NAMES):
                col = valid_vals[:, k]
                record[f"{name}_mean"] = float(col.mean())
                record[f"{name}_std"]  = float(col.std())
                record[f"{name}_max"]  = float(col.max())
                record[f"{name}_min"]  = float(col.min())
        else:
            # Fully black image: default values
            for name in METRIC_NAMES:
                record[f"{name}_mean"] = 0.0
                record[f"{name}_std"]  = 0.0
                record[f"{name}_max"]  = 0.0
                record[f"{name}_min"]  = 0.0

        # Save the full quality map
        np.save(QUALITY_MAPS_DIR / f"{make_image_id(path)}.npy", qmap)

        return record

    except Exception as e:
        return {"image_id": make_image_id(path), "path": str(path), "error": str(e)}


# Parallel processing
records = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as exe:
    futures = {exe.submit(process_one_image, p): p for p in all_paths}
    for future in tqdm(futures, desc="Metric extraction", total=len(all_paths)):
        records.append(future.result())

df_all = pd.DataFrame(records)
n_errors = df_all["error"].notna().sum() if "error" in df_all.columns else 0
print(f"\nImages processed: {len(df_all)}  |  Errors: {n_errors}")
print(f"Colonnes : {list(df_all.columns)}")
df_all.head(3)

## 11. ResNet-18 feature extraction

In [ ]:
if USE_RESNET:
    print("Extraction features ResNet-18...")
    feat_records = []
    for path in tqdm(all_paths, desc="ResNet features"):
        try:
            feat = extract_resnet_features(path)
            feat_records.append({"image_id": make_image_id(path), "resnet_features": feat})
        except Exception as e:
            feat_records.append({"image_id": make_image_id(path), "resnet_features": None})

    # Save separately (numpy array of shape N×512)
    valid   = [(r["image_id"], r["resnet_features"]) for r in feat_records
               if r["resnet_features"] is not None]
    ids_arr  = [v[0] for v in valid]
    feat_arr = np.stack([v[1] for v in valid])  # (N, 512)

    np.save(RESNET_FEAT_DIR / "resnet18_features.npy", feat_arr)
    pd.DataFrame({"image_id": ids_arr}).to_csv(RESNET_FEAT_DIR / "resnet18_ids.csv", index=False)
    print(f"Features saved: {feat_arr.shape}  → outputs/resnet18_features.npy")
else:
    print(f"ResNet extraction disabled (approach {APPROACH})")

## 12. Statistics of the global scores

In [ ]:
mean_cols = [c for c in df_all.columns if c.endswith("_mean")]

print("Statistics of the global metrics (means over patches):")
print(df_all[mean_cols].describe().round(4))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

colors = plt.cm.tab10.colors
for k, (col, ax) in enumerate(zip(mean_cols, axes)):
    data = df_all[col].dropna()
    ax.hist(data, bins=30, color=colors[k % 10], edgecolor="white", alpha=0.85)
    ax.axvline(data.mean(),   color="red",    lw=1.5, ls="--", label="mean")
    ax.axvline(data.median(), color="orange", lw=1.5, ls=":",  label="median")
    ax.set_title(col.replace("_mean", ""), fontsize=9)
    ax.legend(fontsize=7)

for j in range(len(mean_cols), len(axes)):
    axes[j].axis("off")

fig.suptitle(f"Distribution of the IQA metrics — whole dataset ({len(df_all)} images)", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_full_dataset_distributions.png", dpi=100, bbox_inches="tight")
plt.show()

## 13. Correlations between metrics

Check which metrics are redundant and which bring orthogonal information.

In [ ]:
corr = df_all[mean_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

labels = [c.replace("_mean", "") for c in mean_cols]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                fontsize=7, color="black" if abs(corr.values[i, j]) < 0.7 else "white")

ax.set_title("Correlation matrix of the IQA metrics", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_correlation_matrix.png", dpi=100, bbox_inches="tight")
plt.show()

## 14. Top good / bad images according to Tenengrad

The extreme images are identified to feed notebook 3 (manual validation labels).

In [ ]:
df_sorted = df_all.dropna(subset=["tenengrad_mean"]).sort_values("tenengrad_mean")

n_show = 4
worst_paths = [Path(p) for p in df_sorted["path"].head(n_show)]
best_paths  = [Path(p) for p in df_sorted["path"].tail(n_show)]

fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))

for col, (path, score) in enumerate(
    zip(worst_paths, df_sorted["tenengrad_mean"].head(n_show))
):
    axes[0, col].imshow(load_gray(path), cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(f"BAD\n{path.stem}\nTen={score:.4f}", color="crimson", fontsize=8)
    axes[0, col].axis("off")

for col, (path, score) in enumerate(
    zip(best_paths, df_sorted["tenengrad_mean"].tail(n_show))
):
    axes[1, col].imshow(load_gray(path), cmap="gray", vmin=0, vmax=1)
    axes[1, col].set_title(f"GOOD\n{path.stem}\nTen={score:.4f}", color="forestgreen", fontsize=8)
    axes[1, col].axis("off")

fig.suptitle("Extreme images according to the mean Tenengrad", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_extreme_images.png", dpi=120, bbox_inches="tight")
plt.show()

## 15. Export final

In [ ]:
out_csv = METRICS_DIR / "patch_metrics_full.csv"
df_all.drop(columns=["error"], errors="ignore").to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
print(f"Colonnes : {list(df_all.columns)}")
print(f"Quality maps .npy → {QUALITY_MAPS_DIR}/ ({len(list(QUALITY_MAPS_DIR.glob('*.npy')))} files)")

## Summary of notebook 2

| Step | Result |
|---|---|
| Patch extraction | `GRID_SIZE×GRID_SIZE` grid per image |
| 8 metrics per patch | Laplacian, RMS, entropy, Tenengrad (global + 4 directions) |
| 2D quality map | `(G, G, 8)` per image — saved as `.npy` |
| Correlations | Matrix to find the orthogonal metrics |
| ResNet-18 features | 512-D vector — `resnet18_features.npy` (approach C) |
| CSV export | `outputs/patch_metrics_full.csv` |

**Next notebook:** `03_Inductive_Bias_Generation.ipynb` — aggregation of the metrics into a single score $c \in [0, 1]$ through PCA or min-max normalisation, and export of `labels_quality.csv`.